# Phase 22 — Word Embeddings: Word2Vec, GloVe, FastText

Σύγκριση κλασικών word embedding μεθόδων για Food Hazard Detection.

**Μέθοδοι:**
- Word2Vec (trained on our corpus) + SVM
- GloVe (pretrained 100d) + SVM
- FastText (trained on our corpus) + SVM

**Baseline για σύγκριση:**
- TF-IDF + SVM: 0.7419
- Best BERT ensemble: 0.81728

**Σκοπός:** Κατανόηση του gap μεταξύ classical embeddings και transformers.

In [3]:
!pip install gensim -q

'pip' is not recognized as an internal or external command,
operable program or batch file.


In [4]:
import numpy as np
import pandas as pd
import re
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder
from gensim.models import Word2Vec, FastText
from gensim.utils import simple_preprocess
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded')

Libraries loaded


In [5]:
train = pd.read_csv('train.csv')
valid = pd.read_csv('valid.csv')
test  = pd.read_csv('test.csv')

def make_text(df):
    return (df['title'].fillna('') + ' ' + df['text'].fillna('').str[:550]).tolist()

texts_train = make_text(train)
texts_valid = make_text(valid)
texts_test  = make_text(test)

y_train_hazard  = train['hazard-category']
y_train_product = train['product-category']
y_valid_hazard  = valid['hazard-category']
y_valid_product = valid['product-category']

print(f'Train: {len(train)} | Valid: {len(valid)} | Test: {len(test)}')

Train: 5082 | Valid: 565 | Test: 997


In [6]:
# Tokenization για Word2Vec/FastText
def tokenize(texts):
    return [simple_preprocess(t, deacc=True) for t in texts]

tokens_train = tokenize(texts_train)
tokens_valid = tokenize(texts_valid)
tokens_test  = tokenize(texts_test)

print(f'Παράδειγμα tokens: {tokens_train[0][:10]}')


def official_st1_score(y_true_hazard, y_pred_hazard,
                       y_true_product, y_pred_product, verbose=True):
    f1_hazard = f1_score(y_true_hazard, y_pred_hazard, average='macro', zero_division=0)
    mask = (np.array(y_true_hazard) == np.array(y_pred_hazard))
    f1_product = f1_score(
        np.array(y_true_product)[mask],
        np.array(y_pred_product)[mask],
        average='macro', zero_division=0
    ) if mask.sum() > 0 else 0.0
    score = (f1_hazard + f1_product) / 2
    if verbose:
        print(f'  macro-F1 Hazard:                    {f1_hazard:.4f}')
        print(f'  Σωστά hazard predictions:           {mask.sum()}/{len(mask)} ({100*mask.mean():.1f}%)')
        print(f'  macro-F1 Product (given correct h): {f1_product:.4f}')
        print(f'  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━')
        print(f'  OFFICIAL ST1 SCORE:                 {score:.4f}')
    return score


def get_doc_vector(tokens, model, vector_size):
    """Μέσος όρος word vectors για ένα document."""
    vectors = [model.wv[w] for w in tokens if w in model.wv]
    if len(vectors) == 0:
        return np.zeros(vector_size)
    return np.mean(vectors, axis=0)


def evaluate_embeddings(X_train, X_valid, name):
    """Εκπαίδευση SVM και αξιολόγηση στο validation set."""
    print(f'\n=== {name} ===')

    # Hazard classifier
    clf_h = LinearSVC(C=0.5, class_weight='balanced', max_iter=2000, random_state=42)
    clf_h.fit(X_train, y_train_hazard)
    preds_hazard = clf_h.predict(X_valid)

    # Product classifier
    clf_p = LinearSVC(C=0.5, class_weight='balanced', max_iter=2000, random_state=42)
    clf_p.fit(X_train, y_train_product)
    preds_product = clf_p.predict(X_valid)

    score = official_st1_score(
        y_valid_hazard, preds_hazard,
        y_valid_product, preds_product
    )
    return score

Παράδειγμα tokens: ['recall', 'notification', 'fsis', 'case', 'number', 'date', 'opened', 'date', 'closed', 'recall']


## 1. Word2Vec + SVM

Εκπαιδεύουμε Word2Vec στο train corpus μας.
Κάθε document αναπαρίσταται ως ο **μέσος όρος** των word vectors του.

In [7]:
VECTOR_SIZE = 100
WINDOW      = 5
MIN_COUNT   = 2
EPOCHS      = 10

print('Εκπαίδευση Word2Vec...')
w2v_model = Word2Vec(
    sentences=tokens_train,
    vector_size=VECTOR_SIZE,
    window=WINDOW,
    min_count=MIN_COUNT,
    workers=4,
    epochs=EPOCHS,
    seed=42
)
print(f'Vocabulary size: {len(w2v_model.wv):,} words')
print(f'Vector size: {VECTOR_SIZE}d')

# Document vectors
X_train_w2v = np.array([get_doc_vector(t, w2v_model, VECTOR_SIZE) for t in tokens_train])
X_valid_w2v = np.array([get_doc_vector(t, w2v_model, VECTOR_SIZE) for t in tokens_valid])

score_w2v = evaluate_embeddings(X_train_w2v, X_valid_w2v, 'Word2Vec + SVM')

Εκπαίδευση Word2Vec...
Vocabulary size: 9,332 words
Vector size: 100d

=== Word2Vec + SVM ===
  macro-F1 Hazard:                    0.6299
  Σωστά hazard predictions:           483/565 (85.5%)
  macro-F1 Product (given correct h): 0.2959
  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  OFFICIAL ST1 SCORE:                 0.4629


## 2. GloVe + SVM

Χρησιμοποιούμε **pretrained** GloVe vectors (6B tokens, 100d).
Το πλεονέκτημα: έχουν εκπαιδευτεί σε τεράστιο corpus.
Το μειονέκτημα: δεν γνωρίζουν domain-specific λέξεις.

In [10]:
import gensim.downloader as api

print('Κατέβασμα GloVe vectors μέσω gensim...')
glove_model = api.load('glove-wiki-gigaword-100')  # 100d, ~128MB
print(f'Φορτώθηκαν {len(glove_model):,} GloVe vectors')

def get_glove_doc_vector(tokens, model, vector_size=100):
    vectors = [model[w] for w in tokens if w in model]
    if len(vectors) == 0:
        return np.zeros(vector_size)
    return np.mean(vectors, axis=0)

X_train_glove = np.array([get_glove_doc_vector(t, glove_model) for t in tokens_train])
X_valid_glove = np.array([get_glove_doc_vector(t, glove_model) for t in tokens_valid])

score_glove = evaluate_embeddings(X_train_glove, X_valid_glove, 'GloVe (100d) + SVM')

Κατέβασμα GloVe vectors μέσω gensim...
[==================================================] 100.0% 128.1/128.1MB downloaded
Φορτώθηκαν 400,000 GloVe vectors

=== GloVe (100d) + SVM ===
  macro-F1 Hazard:                    0.5626
  Σωστά hazard predictions:           466/565 (82.5%)
  macro-F1 Product (given correct h): 0.3610
  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  OFFICIAL ST1 SCORE:                 0.4618


## 3. FastText + SVM

Το FastText εκπαιδεύεται σε **subword** level (character n-grams).
Πλεονέκτημα: χειρίζεται καλύτερα άγνωστες λέξεις και typos.

In [11]:
print('Εκπαίδευση FastText...')
ft_model = FastText(
    sentences=tokens_train,
    vector_size=VECTOR_SIZE,
    window=WINDOW,
    min_count=MIN_COUNT,
    workers=4,
    epochs=EPOCHS,
    seed=42
)
print(f'Vocabulary size: {len(ft_model.wv):,} words')
print(f'Vector size: {VECTOR_SIZE}d')

# Document vectors
X_train_ft = np.array([get_doc_vector(t, ft_model, VECTOR_SIZE) for t in tokens_train])
X_valid_ft = np.array([get_doc_vector(t, ft_model, VECTOR_SIZE) for t in tokens_valid])

score_ft = evaluate_embeddings(X_train_ft, X_valid_ft, 'FastText + SVM')

Εκπαίδευση FastText...
Vocabulary size: 9,332 words
Vector size: 100d

=== FastText + SVM ===
  macro-F1 Hazard:                    0.6772
  Σωστά hazard predictions:           481/565 (85.1%)
  macro-F1 Product (given correct h): 0.2660
  ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  OFFICIAL ST1 SCORE:                 0.4716


## 4. Σύγκριση Όλων των Μεθόδων

In [12]:
print('\n' + '='*55)
print('ΣΥΓΚΡΙΣΗ ΜΕΘΟΔΩΝ (validation ST1 score)')
print('='*55)

results = [
    ('Word2Vec + SVM',              score_w2v,   'Classical'),
    ('GloVe (pretrained 100d) + SVM', score_glove, 'Classical'),
    ('FastText + SVM',              score_ft,    'Classical'),
    ('TF-IDF + SVM',                0.7419,      'Classical (Kaggle)'),
    ('DistilBERT (train+valid)',     0.7606,      'Transformer (Kaggle)'),
    ('BERT-base + Focal Loss',       0.8040,      'Transformer (Kaggle)'),
    ('BERT-base + Multi-task (0.5/0.5)', 0.8173,  'Transformer (Kaggle)'),
]

results.sort(key=lambda x: x[1])

for name, score, category in results:
    bar = '█' * int(score * 40)
    print(f'{name:40s} {score:.4f}  {bar}')

print('\n=== ΣΥΜΠΕΡΑΣΜΑΤΑ ===')
print(f'Καλύτερο classical embedding: {max(score_w2v, score_glove, score_ft):.4f}')
print(f'TF-IDF + SVM (baseline):       0.7419')
print(f'Best Transformer:              0.8173')
print(f'Gap classical→transformer:     {0.8173 - max(score_w2v, score_glove, score_ft):.4f}')


ΣΥΓΚΡΙΣΗ ΜΕΘΟΔΩΝ (validation ST1 score)
GloVe (pretrained 100d) + SVM            0.4618  ██████████████████
Word2Vec + SVM                           0.4629  ██████████████████
FastText + SVM                           0.4716  ██████████████████
TF-IDF + SVM                             0.7419  █████████████████████████████
DistilBERT (train+valid)                 0.7606  ██████████████████████████████
BERT-base + Focal Loss                   0.8040  ████████████████████████████████
BERT-base + Multi-task (0.5/0.5)         0.8173  ████████████████████████████████

=== ΣΥΜΠΕΡΑΣΜΑΤΑ ===
Καλύτερο classical embedding: 0.4716
TF-IDF + SVM (baseline):       0.7419
Best Transformer:              0.8173
Gap classical→transformer:     0.3457


1. **Word Embeddings << TF-IDF — Το TF-IDF (0.7419)** 
  - κερδίζει κατά πολύ τα Word2Vec/GloVe/FastText (~0.46). Αυτό εξηγείται γιατί το TF-IDF εκμεταλλεύεται συγκεκριμένες λέξεις-κλειδιά (π.χ. "salmonella", "allergen") που είναι πολύ διαγνωστικές για το task, ενώ τα embeddings κάνουν average και χάνουν αυτή την πληροφορία.

2. **FastText > Word2Vec > GloVe** 
  - Το FastText κερδίζει ελαφρά λόγω subword modeling — χειρίζεται καλύτερα τεχνικές λέξεις του food safety domain.

3. **Transformer gap = 0.3457** 
  - Τεράστιο gap μεταξύ classical και transformers, που δείχνει ότι το contextualized representation είναι κρίσιμο για αυτό το task.